In [9]:
# Create transaction-level customer spending data
import pandas as pd
transactions = pd.DataFrame({
    "customer_id": ["C01", "C01", "C02", "C02", "C03", "C03", "C03"],
    "segment": ["new", "new", "returning", "returning", "new", "new", "new"],
    "amount": [40, 60, 120, 80, 20, 30, 50],
})
transactions

,customer_id,segment,amount
0,C01,new,40
1,C01,new,60
2,C02,returning,120
3,C02,returning,80
4,C03,new,20
5,C03,new,30
6,C03,new,50


In [5]:
# Aggregate total spending once per customer
customer_total_spending=transactions.groupby("customer_id",as_index=False)["amount"].sum().rename(columns={"amount":"total_expenditure"
})
customer_total_spending

,customer_id,total_expenditure
0,C01,100
1,C02,200
2,C03,100


In [10]:
# Broadcast each customer total back to every transaction row
transactions["total_spending"]=transactions.groupby("customer_id")["amount"].transform("sum")
transactions

,customer_id,segment,amount,total_spending
0,C01,new,40,100
1,C01,new,60,100
2,C02,returning,120,200
3,C02,returning,80,200
4,C03,new,20,100
5,C03,new,30,100
6,C03,new,50,100


In [11]:
# Calculate each transaction's share of the customer total
transactions["%_customer_total"]=(transactions["amount"]/transactions["total_spending"])*100
transactions

,customer_id,segment,amount,total_spending,%_customer_total
0,C01,new,40,100,40.0
1,C01,new,60,100,60.0
2,C02,returning,120,200,60.0
3,C02,returning,80,200,40.0
4,C03,new,20,100,20.0
5,C03,new,30,100,30.0
6,C03,new,50,100,50.0


In [19]:
# Identify each customer's largest transaction
customer_largest_transaction=transactions.loc[transactions.groupby("customer_id")["amount"].idxmax()]
transactions["is_largest"]=transactions["amount"]==transactions.groupby("customer_id")["amount"].transform("max")
transactions

,customer_id,segment,amount,total_spending,%_customer_total,is_largest
0,C01,new,40,100,40.0,False
1,C01,new,60,100,60.0,True
2,C02,returning,120,200,60.0,True
3,C02,returning,80,200,40.0,False
4,C03,new,20,100,20.0,False
5,C03,new,30,100,30.0,False
6,C03,new,50,100,50.0,True


In [17]:
# Compare transactions with the overall average amount
avg_spending=transactions["amount"].mean()
above_avg_spending_customer=transactions[transactions["amount"]>avg_spending]
above_avg_spending_customer

,customer_id,segment,amount,total_spending,%_customer_total
1,C01,new,60,100,60.0
2,C02,returning,120,200,60.0
3,C02,returning,80,200,40.0


### Important notes

- `groupby().sum()` reduces to one row per customer, while `transform("sum")` preserves the original transaction rows.
- The percentage calculation should sum to approximately 100% for each customer, subject to rounding.
- `idxmax()` returns the first transaction when a customer has tied maximum amounts.
- Missing amounts should be handled before calculating totals or percentages.
- The overall average is different from each customer's average; choose the baseline that matches the business question.
